# CNN Autoencoder for Anomaly Detection Using Toy Car Audio

**Purpose:** This notebook presents a complete pipeline for toy car anomaly detection based on operating sound, including data loading, preprocessing, CNN autoencoder training, and result analysis.

## 1. Imports

In [ ]:
import os 
import glob 
import random
from collections import defaultdict, Counter

import numpy as np

import librosa
import librosa.display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader

from sklearn.metrics import average_precision_score

import pickle

import optuna
import optuna.visualization as vis

import matplotlib.pyplot as plt

from concurrent.futures import ThreadPoolExecutor, as_completed

# Shared helpers for this notebook and future model notebooks
from helpers.helper_audio_data import (
    AnomalousDataset,
    NormalDataset,
    crop_audio,
    load_all_wav_paths,
    structure_wav_paths,
    wav_to_logmel,
)
from helpers.helper_npy_data import (
    UnifiedNPYDataset,
    compute_target_T_from_npy,
    convert_to_logmel,
    crop_mel,
    load_precomputed_samples,
    reduce_samples,
    sec_to_frame,
    split_cnt_to_segments,
    split_indices,
)
from helpers.helper_eval import (
    build_scope_loader_dict,
    count_ind_cnt,
    downsample_anomaly_scores,
    evaluate_scores,
    find_best_f1_threshold,
    get_reconstruction_scores,
    infer_case_from_path,
    plot_multichannel_spec,
    plot_one_reconstruction,
)

## 2. Paths and configuration

In [ ]:
# Paths

DATA_BASE_PATH = "F:ToyCar"

BEST_MODEL_SAVE_PATH = "best_Transformer_AE.pth"

In [ ]:
# Configs

CASES = ['case1', 'case2', 'case3', 'case4']
CHANNELS = ['1', '2', '3', '4']
SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 3. Data Loading

This section uses reusable utilities from `helpers/helper_audio_data.py` so the same data pipeline can be shared across other model notebooks.

In [ ]:
# Data loading helpers are imported from `helpers/helper_audio_data.py`.
# Keeping this notebook focused on training/evaluation flow improves readability.
#
# Available helpers in scope:
# - load_all_wav_paths(...)
# - structure_wav_paths(...)

In [ ]:
# Spectrogram utilities are imported from `helpers/helper_audio_data.py`.
# - crop_audio(...)
# - wav_to_logmel(...)

In [ ]:
# `NormalDataset` is imported from `helpers/helper_audio_data.py`.
# It handles:
# 1) grouping per (case, sample_id)
# 2) channel completeness checks
# 3) waveform -> normalized log-mel conversion
# 4) per-sample temporal alignment across channels

In [ ]:
# `AnomalousDataset` is imported from `helpers/helper_audio_data.py`.
# It reuses the same grouped multi-channel logic as `NormalDataset`,
# with anomaly-specific filename parsing handled in the helper.

In [ ]:
# Build both datasets with a consistent crop window to keep training input length stable.
CROP_START_SEC = 1.5
CROP_END_SEC = 7.7

normal_dataset = NormalDataset(
    DATA_BASE_PATH,
    cases=CASES,
    channels=CHANNELS,
    start_sec=CROP_START_SEC,
    end_sec=CROP_END_SEC,
)

anom_dataset = AnomalousDataset(
    DATA_BASE_PATH,
    cases=CASES,
    channels=CHANNELS,
    start_sec=CROP_START_SEC,
    end_sec=CROP_END_SEC,
)

print(f"Anomalous samples: {len(anom_dataset)}")
print(f"Normal samples: {len(normal_dataset)}")

## 4. Data Preprocessing

Reusable preprocessing and dataset logic is moved to helper modules so this section stays focused on how to run each step.

In [ ]:
# Reusable NPY/wav preprocessing helpers are imported from
# `helpers/helper_npy_data.py` to keep this notebook concise.
#
# Available helpers in scope:
# - convert_to_logmel(...)
# - load_precomputed_samples(...)
# - reduce_samples(...)
# - sec_to_frame(...)
# - crop_mel(...)

In [ ]:
# Optional preprocessing step:
# split CNT spectrogram files into fixed-length segments.
# Keep this as a one-liner for readability; implementation lives in helpers.
split_cnt_to_segments(DATA_BASE_PATH, max_workers=8)

In [ ]:
# `UnifiedNPYDataset` is imported from `helpers/helper_npy_data.py`.
# It unifies IND/CNT loading, balances CNT windows, and supports
# optional anomaly inclusion for validation/testing.

In [ ]:
# Estimate a robust common length from IND normal data, then build the unified dataset.
target_T = compute_target_T_from_npy(
    base_path=f"{DATA_BASE_PATH}/npy",
    cases=CASES,
)

npy_dataset = UnifiedNPYDataset(
    base_path=f"{DATA_BASE_PATH}/npy",
    cases=CASES,
    target_T=target_T,
    use_ind=True,
    use_cnt=True,
    include_anomaly=True,
    stride_ratio=5.0,
)

print(f"Dataset size: {len(npy_dataset)} | target_T: {target_T}")

In [ ]:
random.seed(42)

# -------------------------
# Separate indices
# -------------------------
normal_indices = [i for i, (_, _, y) in enumerate(npy_dataset.samples) if y == 0]
anom_indices   = [i for i, (_, _, y) in enumerate(npy_dataset.samples) if y == 1]

# split normal into IND / CNT
ind_indices = [i for i in normal_indices if npy_dataset.samples[i][1] is None]
cnt_indices = [i for i in normal_indices if npy_dataset.samples[i][1] is not None]

# -------------------------
# Shuffle separately
# -------------------------
random.shuffle(ind_indices)
random.shuffle(cnt_indices)

# -------------------------
# Stratified split helper
# -------------------------
# split_indices(...) is imported from `helpers/helper_npy_data.py`
# to keep split policy consistent across notebooks.

# -------------------------
# Stratified split
# -------------------------
ind_train, ind_val, ind_test = split_indices(ind_indices)
cnt_train, cnt_val, cnt_test = split_indices(cnt_indices)

# merge back
train_idx        = ind_train + cnt_train
val_normal_idx   = ind_val + cnt_val
test_normal_idx  = ind_test + cnt_test

# shuffle merged splits
random.shuffle(train_idx)
random.shuffle(val_normal_idx)
random.shuffle(test_normal_idx)

# -------------------------
# ANOMALY split (50 / 50)
# -------------------------
random.shuffle(anom_indices)

m = len(anom_indices)
val_anom_idx  = anom_indices[:m // 2]
test_anom_idx = anom_indices[m // 2:]

# -------------------------
# DATASETS
# -------------------------
train_dataset = Subset(npy_dataset, train_idx)

val_normal = Subset(npy_dataset, val_normal_idx)
val_anom   = Subset(npy_dataset, val_anom_idx)

test_normal = Subset(npy_dataset, test_normal_idx)
test_anom   = Subset(npy_dataset, test_anom_idx)

In [ ]:
# Quick distribution check for IND vs CNT samples in each split.
train_ind, train_cnt = count_ind_cnt(train_dataset, npy_dataset.samples)
val_ind, val_cnt = count_ind_cnt(val_normal, npy_dataset.samples)
test_ind, test_cnt = count_ind_cnt(test_normal, npy_dataset.samples)

print("Train Normal  -> IND:", train_ind, "CNT:", train_cnt)
print("Val Normal    -> IND:", val_ind, "CNT:", val_cnt)
print("Test Normal   -> IND:", test_ind, "CNT:", test_cnt)
print("Val/Test anomaly sizes:", len(val_anom), len(test_anom))

In [ ]:
# Baseline loaders from direct split (kept for quick sanity checks).
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_normal_loader = DataLoader(val_normal, batch_size=8, shuffle=False)
val_anom_loader = DataLoader(val_anom, batch_size=8, shuffle=False)
test_normal_loader = DataLoader(test_normal, batch_size=8, shuffle=False)
test_anom_loader = DataLoader(test_anom, batch_size=8, shuffle=False)

In [ ]:
# -------------------------
# CASE-AWARE LOADERS
# -------------------------
sample_cases = [infer_case_from_path(p) for (p, _, _) in npy_dataset.samples]

VAL_CASE_SCOPES = {
    "case1_only": [CASES[0]],
    "case2_only": [CASES[1]],
    "case3_only": [CASES[2]],
    "case4_only": [CASES[3]],
    "all_cases": CASES,
}

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

val_loaders = build_scope_loader_dict(
    dataset=npy_dataset,
    sample_cases=sample_cases,
    normal_indices=val_normal_idx,
    anom_indices=val_anom_idx,
    case_scopes=VAL_CASE_SCOPES,
    batch_size=8,
    anom_ratio=0.05,
    prefix="val",
)

for scope_name, data in val_loaders.items():
    print(f"[VAL] {scope_name}: normal={data['n_normal']}, anomaly={data['n_anom']}")

test_loaders = build_scope_loader_dict(
    dataset=npy_dataset,
    sample_cases=sample_cases,
    normal_indices=test_normal_idx,
    anom_indices=test_anom_idx,
    case_scopes=VAL_CASE_SCOPES,
    batch_size=8,
    anom_ratio=0.05,
    prefix="test",
)

for scope_name, data in test_loaders.items():
    print(f"[TEST] {scope_name}: normal={data['n_normal']}, anomaly={data['n_anom']}")

val_normal_loader = val_loaders["all_cases"]["val_normal_loader"]
val_anom_loader = val_loaders["all_cases"]["val_anom_loader"]
test_normal_loader = test_loaders["all_cases"]["test_normal_loader"]
test_anom_loader = test_loaders["all_cases"]["test_anom_loader"]

## 6. Model

In [ ]:
class TransformerAutoEncoder(nn.Module):
    def __init__(
        self,
        input_shape=(4, 64, 64),
        time_patch=8,
        emb_dim=128,
        depth=3,
        num_heads=4,
        latent_dim=32,
        dropout=0.1
    ):
        super().__init__()

        C, H, W = input_shape
        self.input_shape = input_shape
        self.time_patch = time_patch

        self.patch_dim = C * H * time_patch

        # ---------------------
        # Patch embedding (time only)
        # ---------------------
        self.patch_embed = nn.Linear(self.patch_dim, emb_dim)

        self.pos_embed = nn.Parameter(torch.randn(1, 1000, emb_dim))  # max length buffer

        # ---------------------
        # Transformer encoder
        # ---------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # ---------------------
        # Bottleneck
        # ---------------------
        self.fc_enc = nn.Linear(emb_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, emb_dim)

        # ---------------------
        # Transformer decoder
        # ---------------------
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=depth)

        # ---------------------
        # Reconstruction
        # ---------------------
        self.output_proj = nn.Linear(emb_dim, self.patch_dim)

    def patchify(self, x):
        B, C, H, W = x.shape
        p = self.time_patch

        W_new = (W // p) * p
        x = x[:, :, :, :W_new]

        x = x.unfold(3, p, p)  # (B, C, H, N, p)
        x = x.permute(0, 3, 1, 2, 4)  # (B, N, C, H, p)
        x = x.reshape(B, -1, self.patch_dim)

        return x, W_new

    def unpatchify(self, x, W_new):
        B, N, _ = x.shape
        p = self.time_patch
        C, H = self.input_shape[0], self.input_shape[1]

        x = x.view(B, N, C, H, p)
        x = x.permute(0, 2, 3, 1, 4)
        x = x.reshape(B, C, H, W_new)

        return x

    def forward(self, x):
        patches, W_new = self.patchify(x)

        N = patches.shape[1]

        z = self.patch_embed(patches)
        z = z + self.pos_embed[:, :N, :]

        z = self.encoder(z)

        z = self.fc_enc(z)
        z = self.fc_dec(z)

        z = self.decoder(z)

        patches = self.output_proj(z)

        out = self.unpatchify(patches, W_new)

        return F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)

## 7. Fine Tuning

In [ ]:
TUNE_SCOPE = "all_cases"
N_TRIALS = 30
TUNE_EPOCHS = 20

def objective(trial):
    print(f"\n=== Trial {trial.number} ===")

    # ======================
    # TRANSFORMER PARAMS
    # ======================
    time_patch = trial.suggest_categorical("time_patch", [4, 8, 16])
    emb_dim = trial.suggest_categorical("emb_dim", [64, 128])
    trans_depth = trial.suggest_int("trans_depth", 2, 4)
    num_heads = trial.suggest_categorical("num_heads", [2, 3])

    if emb_dim % num_heads != 0:
        raise optuna.TrialPruned()

    latent_dim = trial.suggest_categorical("latent_dim", [8, 16, 32])

    # ======================
    # TRAINING PARAMS
    # ======================
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)

    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    noise_std = trial.suggest_float("noise_std", 0.0, 0.05)

    score_type = trial.suggest_categorical("score_type", ["l1", "l2"])

    # ======================
    # DATA
    # ======================
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sample_x, _ = next(iter(train_loader))
    input_shape = sample_x.shape[1:]

    # ======================
    # MODEL
    # ======================
    model = TransformerAutoEncoder(
        input_shape=input_shape,
        time_patch=time_patch,
        emb_dim=emb_dim,
        depth=trans_depth,
        num_heads=num_heads,
        latent_dim=latent_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if score_type == "l1":
        criterion = nn.L1Loss()
    else:
        criterion = nn.MSELoss()

    # ======================
    # HELPER
    # ======================
    def compute_pr_auc(vn_loader, va_loader):
        normal_scores = get_reconstruction_scores(model, vn_loader, device, score_type=score_type)
        anom_scores = get_reconstruction_scores(model, va_loader, device, score_type=score_type)

        y_true = np.concatenate([
            np.zeros(len(normal_scores)),
            np.ones(len(anom_scores))
        ])
        y_score = np.concatenate([normal_scores, anom_scores])

        if len(np.unique(y_true)) < 2:
            return 0.5, normal_scores, anom_scores

        return average_precision_score(y_true, y_score), normal_scores, anom_scores

    # ======================
    # TRAIN
    # ======================
    for epoch in range(TUNE_EPOCHS):
        model.train()
        total_loss = 0

        for x, _ in train_loader:
            x = x.to(device)

            if noise_std > 0:
                x_noisy = x + noise_std * torch.randn_like(x)
            else:
                x_noisy = x

            recon = model(x_noisy)
            loss = criterion(recon, x)

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            vn_loader = val_loaders[TUNE_SCOPE]["val_normal_loader"]
            va_loader = val_loaders[TUNE_SCOPE]["val_anom_loader"]

            pr_auc, _, _ = compute_pr_auc(vn_loader, va_loader)

            if epoch >= 10:
                trial.report(pr_auc, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

    # ======================
    # FINAL VALIDATION
    # ======================
    results = {}

    for scope_name, loaders in val_loaders.items():
        vn_loader = loaders["val_normal_loader"]
        va_loader = loaders["val_anom_loader"]

        pr_auc, normal_scores, anom_scores = compute_pr_auc(vn_loader, va_loader)

        results[scope_name] = pr_auc
        trial.set_user_attr(f"{scope_name}_pr_auc", float(pr_auc))

        if scope_name == TUNE_SCOPE:
            gap = float(np.mean(anom_scores) - np.mean(normal_scores))
            trial.set_user_attr("gap", gap)

    print("Validation PR AUC:")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")

    return float(results[TUNE_SCOPE])

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)
def print_callback(study, trial):
    print(f"\nTrial {trial.number} finished")
    if trial.value is None:
        print("[ERROR] Failed!")
    else:
        print(f"  Value: {trial.value:.4f}")
    print(f"  Params: {trial.params}")

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_callback])

for t in study.trials:
    print_callback(study, t)

print("[BEST] AUC:", study.best_value)
print("[BEST] params:", study.best_params)
print("[BEST] gap:", study.best_trial.user_attrs.get("gap", None))

In [ ]:

# ================================
# Optimization History
# ================================
# Shows how the objective value (PR-AUC) evolves over trials.
# Useful for checking:
# - whether the search is converging
# - whether more trials are needed
# - stability of the optimization process
vis.plot_optimization_history(study).show()


# ================================
# Hyperparameter Importance
# ================================
# Ranks hyperparameters based on their impact on the objective.
# Useful for:
# - identifying which parameters matter most (e.g., lr, latent_dim)
# - pruning irrelevant parts of the search space
vis.plot_param_importances(study).show()


## 8. Training the Best Model

In [ ]:
# Rebuild model with best params
best_params = study.best_params
time_patch=best_params['time_patch']
emb_dim=best_params['emb_dim']
trans_depth=best_params['trans_depth']
num_heads=best_params['num_heads']
latent_dim=best_params['latent_dim']
dropout=best_params['dropout']

sample_x, _ = next(iter(train_loader))
input_shape = sample_x.shape[1:]
model = TransformerAutoEncoder(
    input_shape=input_shape,
    time_patch=time_patch,
    emb_dim=emb_dim,
    depth=trans_depth,
    num_heads=num_heads,
    latent_dim=latent_dim,
    dropout=dropout
).to(device)

In [ ]:
# Train final model

train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

criterion = nn.L1Loss()

for epoch in range(TUNE_EPOCHS):
    model.train()
    for x, _ in train_loader:
        x = x.to(device)

        if best_params["noise_std"] > 0:
            x_noisy = x + best_params["noise_std"] * torch.randn_like(x)
        else:
            x_noisy = x

        recon = model(x_noisy)
        loss = criterion(recon, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)

## 9. Model Evaluation

In [ ]:
# Compute validation threshold (maximize F1 on validation set).
val_normal_scores = get_reconstruction_scores(
    model,
    val_normal_loader,
    device,
    score_type=best_params["score_type"],
)
val_anom_scores = get_reconstruction_scores(
    model,
    val_anom_loader,
    device,
    score_type=best_params["score_type"],
)

best_threshold = find_best_f1_threshold(val_normal_scores, val_anom_scores)
print("Best threshold:", best_threshold)

In [ ]:
# Evaluate on the overall test scope.
test_normal_scores = get_reconstruction_scores(
    model,
    test_normal_loader,
    device,
    score_type=best_params["score_type"],
)
test_anom_scores = get_reconstruction_scores(
    model,
    test_anom_loader,
    device,
    score_type=best_params["score_type"],
)

# Keep anomaly ratio aligned with validation protocol.
test_anom_scores = downsample_anomaly_scores(
    test_normal_scores,
    test_anom_scores,
    anom_ratio=0.05,
)

pr_auc, f1, gap = evaluate_scores(test_normal_scores, test_anom_scores, best_threshold)

print("Best threshold:", best_threshold)
print("Test PR-AUC:", pr_auc)
print("Test F1:", f1)
print("Normal Mean Error:", float(np.mean(test_normal_scores)))
print("Anomalous Mean Error:", float(np.mean(test_anom_scores)))
print("Error Gap:", gap)

In [ ]:
# Per-case test results
print("\n=== Per-case Test Results ===")

all_normal_scores = []
all_anom_scores = []

for scope_name, loaders in test_loaders.items():
    normal_scores = get_reconstruction_scores(
        model,
        loaders["test_normal_loader"],
        device,
        score_type=best_params["score_type"],
    )
    anom_scores = get_reconstruction_scores(
        model,
        loaders["test_anom_loader"],
        device,
        score_type=best_params["score_type"],
    )

    # Use full anomaly set per case for stable per-case reporting.
    anom_scores = downsample_anomaly_scores(normal_scores, anom_scores, anom_ratio=1.0)

    if scope_name != "all_cases":
        all_normal_scores.extend(normal_scores)
        all_anom_scores.extend(anom_scores)

    pr_auc, f1, gap = evaluate_scores(normal_scores, anom_scores, best_threshold)
    print(f"{scope_name}: PR-AUC={pr_auc:.4f}, F1={f1:.4f}, Gap={gap:.4f}")

## 10. Result visualization

In [ ]:
# Visualization helpers are imported from `helpers/helper_eval.py`:
# - plot_multichannel_spec(...)
# - plot_one_reconstruction(...)

In [ ]:
test_anom_iter = iter(test_anom_loader)
plot_one_reconstruction(model, test_anom_iter, device)

In [ ]:
test_normal_iter = iter(test_normal_loader)
plot_one_reconstruction(model, test_normal_iter, device)